# IOAI — 2026 Local Mock Paintings (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-paintings.zip', '/tmp/paint.zip')
    with zipfile.ZipFile('/tmp/paint.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Paintings — 그림 진위/화가군집/가격 (RoAI/ONIA 2026 Mock)

그림 속성 표로 3 서브태스크:
1. (결정적) Artistic Authenticity Score 규칙 계산 → 'Autentic'/'Incert'
2. (군집) 5명 화가로 KMeans 군집 (k=5)
3. (ML) `target_price` 회귀 — MAE 로 채점

**제출**: `submission.csv` 롱포맷 `SampleID, subtaskID, Answer` (subtaskID=`Task1/Task2/Task3`).
**채점(로컬)**: ST1 exact, ST2 레퍼런스 군집 대비 ARI, ST3 train held-out MAE.
**데이터**: `data/train.csv`(+`data/test.csv`). 원본: platform.olimpiada-ai.ro/competitions/7

In [ ]:
import pandas as pd, numpy as np
from sklearn.linear_model import LinearRegression

root_path = "data"
train_df = pd.read_csv(f"{root_path}/train.csv")
test_df = pd.read_csv(f"{root_path}/test.csv")
train_df.head()

In [ ]:
# Task3 (ML, 채점 대상): 가격 회귀 — 간단 베이스라인(수치형 LinearRegression).
# 모범답안은 canvas_size 분해(면적·비율)·범주 원핫·화가군집 feature·Ridge 로 MAE 를 크게 낮춥니다.
feat = [c for c in train_df.select_dtypes(include=np.number).columns if c not in ("target_price","SampleID")]
mean = train_df[feat].mean()
reg = LinearRegression().fit(train_df[feat].fillna(mean), train_df["target_price"])
subtask3 = reg.predict(test_df[feat].fillna(mean))

In [ ]:
# Task1·Task2 는 채점 헤드라인(Task3 MAE)에 미포함 — 우선 임시값. TODO 로 직접 구현해보세요.
# Task1: Artistic Authenticity Score 규칙 → 'Autentic'/'Incert'
# Task2: 화풍 feature 표준화 후 KMeans(k=5) 화가 군집
subtask1 = ["Incert"] * len(test_df)
subtask2 = [0] * len(test_df)

In [ ]:
# 제출(롱포맷): SampleID, subtaskID(Task1/Task2/Task3), Answer
def build_df(sid, ans):
    return pd.DataFrame({"SampleID": test_df["SampleID"], "subtaskID": sid, "Answer": np.asarray(ans)})
submission_df = pd.concat([build_df("Task1", subtask1), build_df("Task2", subtask2), build_df("Task3", subtask3)])
submission_df.to_csv(f"{root_path}/submission.csv", index=False)
submission_df.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)